EURON 11기 DL 세션 2주차 예습과제
- 4.2.1 활성화 함수 (렐루 + 소프트맥스)
- 4.2.1 손실 함수 (MSE, 크로스 엔트로피)
- 4.2.3 드롭아웃
- 4.2.3 미니 배치 경사 하강법 (Dataset / DataLoader)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(torch.__version__)

2.11.0+cpu


## 1. 활성화 함수 — 렐루 + 소프트맥스 (p.145)

은닉층에는 **렐루(ReLU)**, 출력층에는 **소프트맥스(Softmax)**를 적용한 신경망.

- 렐루: 입력이 음수면 0, 양수면 그대로 출력 → 기울기 소멸 문제 없음, 학습 빠름
- 소프트맥스: 출력 값을 0~1 사이로 정규화, 총합이 1 → 다중 분류 출력층에 사용


In [2]:
class Net(torch.nn.Module):
    def __init__(self, n_feature, n_hidden, n_output):
        super(Net, self).__init__()
        self.hidden = torch.nn.Linear(n_feature, n_hidden)   # 은닉층
        self.relu = torch.nn.ReLU(inplace=True)
        self.out = torch.nn.Linear(n_hidden, n_output)       # 출력층
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)      # 은닉층을 위한 렐루 활성화 함수
        x = self.out(x)
        x = self.softmax(x)   # 출력층을 위한 소프트맥스 활성화 함수
        return x

net = Net(n_feature=4, n_hidden=8, n_output=3)
sample = torch.randn(2, 4)          # 샘플 2개
out = net(sample)
print(out)
print("각 행의 합:", out.sum(dim=1))  # 소프트맥스라 1에 가까워야 함

tensor([[0.2479, 0.4134, 0.3387],
        [0.2094, 0.3465, 0.4441]], grad_fn=<SoftmaxBackward0>)
각 행의 합: tensor([1.0000, 1.0000], grad_fn=<SumBackward1>)


## 2. 손실 함수 ① 평균 제곱 오차 MSE (p.146)

MSE = (1/n) Σ (yᵢ − ŷᵢ)²

- 실제 값과 예측 값의 차이를 제곱해서 평균
- 값이 작을수록 예측력이 좋음
- 주로 **회귀**에서 사용

> `reduction='sum'` : 평균 대신 합으로 계산. 기본값은 `'mean'`

In [3]:
import torch
model=torch.nn.Linear(3,1)
x=torch.randn(5,3)
y=torch.randn(5,1)
loss_fn=torch.nn.MSELoss(reduction='sum')
y_pred=model(x)
loss=loss_fn(y_pred,y)
print(loss)

tensor(4.9158, grad_fn=<MseLossBackward0>)


## 3. 손실 함수 ② 크로스 엔트로피 오차 CEE (p.146~147)

CrossEntropy = −Σ yᵢ log ŷᵢ

- **분류** 문제에서 원-핫 인코딩했을 때 사용
- 시그모이드의 자연 상수 e 때문에 MSE를 쓰면 손실 그래프가 울퉁불퉁 → 로그를 취해 매끈하게 만듦

> 코드 설명
> - `torch.randn(5, 6)` : 평균 0, 표준편차 1인 정규분포에서 숫자 생성 → 샘플 5개, 클래스 6개짜리 출력
> - `torch.empty(5, dtype=torch.long).random_(6)` : 0~5 사이 정수 5개 → 정답 레이블
> - `nn.CrossEntropyLoss()`는 내부에서 소프트맥스 + 로그를 같이 처리하므로 원-핫 대신 정수 레이블을 바로 넣음

In [4]:
loss=nn.CrossEntropyLoss()
input=torch.rand(5, 6, requires_grad=True)
target=torch.empty(5, dtype=torch.long).random_(6)
output=loss(input,target)
print("target:", target)
print("loss:", output)

target: tensor([5, 5, 2, 1, 3])
loss: tensor(1.7444, grad_fn=<NllLossBackward0>)


## 4. 드롭아웃 Dropout (p.150)

과적합을 막기 위해 **학습 중 임의로 일부 노드를 제외**하는 방법.

- `torch.nn.Dropout(0.5)` : 50%의 노드를 무작위로 골라 사용하지 않음
- 784 → 1200 → 1200 → 10 구조 (MNIST 손글씨 28×28=784 픽셀 → 숫자 0~9 분류를 가정)

> 드롭아웃은 **학습 때만** 적용되고, 평가 때(`model.eval()`)는 꺼짐

In [6]:
class DropoutModel(torch.nn.Module):
    def __init__(self):
        super(DropoutModel, self).__init__()
        self.layer1 = torch.nn.Linear(784, 1200)
        self.dropout1 = torch.nn.Dropout(0.5)   # 50%의 노드를 무작위로 선택하여 사용하지 않겠다는 의미
        self.layer2 = torch.nn.Linear(1200, 1200)
        self.dropout2 = torch.nn.Dropout(0.5)
        self.layer3 = torch.nn.Linear(1200, 10)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = self.dropout1(x)
        x = F.relu(self.layer2(x))
        x = self.dropout2(x)
        return self.layer3(x)

# 동작 확인
dropout_model = DropoutModel()
sample = torch.randn(1, 784)

dropout_model.train()               # 학습 모드 → 드롭아웃 켜짐
print("train 모드:", dropout_model(sample)[0, :5])

dropout_model.eval()                # 평가 모드 → 드롭아웃 꺼짐
print("eval 모드 :", dropout_model(sample)[0, :5])


train 모드: tensor([-0.1866,  0.0691, -0.2157, -0.0181,  0.2217], grad_fn=<SliceBackward0>)
eval 모드 : tensor([-0.1513,  0.0335,  0.0088,  0.1793, -0.0358], grad_fn=<SliceBackward0>)


## 5. 미니 배치 경사 하강법 — Dataset / DataLoader (p.153~154)

전체 데이터를 **미니 배치**로 나눠서 배치마다 기울기를 구하고 평균 기울기로 업데이트.
배치 경사 하강법보다 빠르고, 확률적 경사 하강법보다 안정적 → 실제로 가장 많이 사용.

파이토치에서는 `Dataset`을 상속해 데이터를 정의하고, `DataLoader`로 배치 단위로 꺼냄.

- `__len__` : 데이터 개수
- `__getitem__` : idx번째 데이터를 텐서로 반환
- `batch_size=2` : 미니 배치 크기 (보통 2의 제곱수 사용)
- `shuffle=True` : 불러올 때마다 랜덤으로 섞음

In [8]:
class CustomDataset(Dataset):
  def __init__(self):
        self.x_data = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
        self.y_data = [[12], [18], [11]]
  def __len__(self):
        return len(self.x_data)
  def __getitem__(self, idx):
        x = torch.FloatTensor(self.x_data[idx])
        y = torch.FloatTensor(self.y_data[idx])
        return x, y
dataset=CustomDataset()
dataloader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
)
for i, (x_batch, y_batch) in enumerate(dataloader):
    print(f"배치 {i}: x = {x_batch.tolist()}, y = {y_batch.tolist()}")

배치 0: x = [[1.0, 2.0, 3.0], [7.0, 8.0, 9.0]], y = [[12.0], [11.0]]
배치 1: x = [[4.0, 5.0, 6.0]], y = [[18.0]]


## 6. 옵티마이저 선언 모음 (p.155~157)

교재 Note에 나온 옵티마이저 선언 코드. `model.parameters()`에 위에서 만든 Net을 넣어 실제로 생성해 봄.

| 옵티마이저 | 조정 대상 | lr 기본값 |
| --- | --- | --- |
| Adagrad | 속도 | 1e-2 |
| Adadelta | 속도 | 1.0 |
| RMSprop | 속도 | 1e-2 |
| SGD + momentum | 운동량 | - |
| SGD + nesterov | 운동량 | - |
| Adam | 속도 + 운동량 | 1e-3 |

In [9]:
model= Net(n_feature=4, n_hidden=8, n_output=3)

optimizer=torch.optim.Adagrad(model.parameters(), lr=0.01)
optimizer=torch.optim.Adadelta(model.parameters(), lr=1.0)
optimizer=torch.optim.RMSprop(model.parameters(), lr=0.01, momentum=0.9)
optimizer=torch.optim.SGD(model.parameters(), lr=0.01)
optimizer=torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)
print(optimizer)


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)
